In [1]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
import yfinance as yf

# 한글 폰트 설정 (Windows: Malgun Gothic, Mac: AppleGothic)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# -------------------------------------------------------------------
# 1. 데이터 수집 (삼성전자 2023-01-01 ~ 2024-12-31, 100개 이상 포인트)
# -------------------------------------------------------------------
print('데이터 다운로드 시작...')
df = yf.download('005930.KS', start='2023-01-01', end='2024-12-31')

# yfinance MultiIndex 컬럼 정리
if isinstance(df.columns, pd.MultiIndex):
    df = df['Close'].rename(columns={'005930.KS': 'Close'})
else:
    df = df[['Close']]

df.reset_index(inplace=True)
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)

# 결측치 보충 (전일 값으로 채우기)
df['Close'] = df['Close'].ffill()

# data 폴더에 원본 데이터 csv 저장
os.makedirs('data', exist_ok=True)
df.to_csv('data/samsung_2023_2024.csv')
print('data/samsung_2023_2024.csv 저장 완료!')

# -------------------------------------------------------------------
# 2. 시계열 분석 기법 적용 (이동평균, 수익률, 변동성)
# -------------------------------------------------------------------
df['MA20'] = df['Close'].rolling(window=20).mean()
df['MA60'] = df['Close'].rolling(window=60).mean()
df['Daily_Return'] = df['Close'].pct_change() * 100
df['Volatility_20D'] = df['Daily_Return'].rolling(window=20).std()

# 월별 수익률 집계
monthly_df = df['Close'].resample('ME').last().pct_change() * 100

# -------------------------------------------------------------------
# 3. 시각화 생성 및 images/ 폴더에 저장
# -------------------------------------------------------------------
os.makedirs('images', exist_ok=True)

# 시각화 1: 주가 추이 및 이동평균선
plt.figure(figsize=(12, 6))
plt.plot(df.index, df['Close'], label='종가 (Close)', color='gray', alpha=0.5)
plt.plot(
    df.index, df['MA20'], label='20일 이동평균 (단기)', color='blue', linewidth=1.5
)
plt.plot(
    df.index, df['MA60'], label='60일 이동평균 (중기)', color='orange', linewidth=1.5
)
plt.title('삼성전자 주가 추이 및 이동평균선 (2023~2024)', fontsize=14)
plt.xlabel('날짜')
plt.ylabel('주가 (원)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('images/01_price_ma_trend.png', dpi=300)
plt.close()

# 시각화 2: 월별 수익률 (MoM)
plt.figure(figsize=(12, 5))
colors = ['red' if x > 0 else 'blue' for x in monthly_df.fillna(0)]
monthly_df.plot(kind='bar', color=colors)
plt.title('삼성전자 월별 수익률 (MoM, %)', fontsize=14)
plt.xlabel('월 (Year-Month)')
plt.ylabel('수익률 (%)')
plt.xticks(
    range(len(monthly_df)),
    [d.strftime('%Y-%m') for d in monthly_df.index],
    rotation=45,
)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('images/02_monthly_returns.png', dpi=300)
plt.close()

# 시각화 3: 20일 이동 표준편차 (변동성 추이)
plt.figure(figsize=(12, 5))
plt.plot(df.index, df['Volatility_20D'], color='purple', label='20일 변동성(표준편차)')
plt.title('삼성전자 일별 수익률의 20일 변동성 추이', fontsize=14)
plt.xlabel('날짜')
plt.ylabel('표준편차 (%)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('images/03_volatility_trend.png', dpi=300)
plt.close()

# -------------------------------------------------------------------
# 4. 보너스: 시계열 분해 (안전성 강화 코드 적용)
# -------------------------------------------------------------------
try:
    # 'Close' 컬럼을 1차원 Series 및 float 형으로 명확히 정제
    close_series = df['Close']
    if isinstance(close_series, pd.DataFrame):
        close_series = close_series.iloc[:, 0]

    close_series = close_series.dropna().astype(float)

    # 시계열 분해 실행 (영업일 기준 약 1개월 = period=20)
    decomposition = seasonal_decompose(
        close_series, model='additive', period=20
    )

    # 그래프 생성 및 크기/레이아웃 조정
    fig = decomposition.plot()
    fig.set_size_inches(12, 8)
    fig.suptitle(
        '삼성전자 주가 시계열 분해 (Trend / Seasonal / Resid)',
        fontsize=14,
        y=1.02,
    )

    plt.tight_layout()
    plt.savefig('images/04_stl_decomposition.png', dpi=300, bbox_inches='tight')
    plt.close(fig)

    print('images/04_stl_decomposition.png 저장 성공!')

except Exception as e:
    print(f'시계열 분해 처리 중 오류 발생: {e}')

print('모든 그래프 생성 및 images/ 폴더 저장 완료!')

데이터 다운로드 시작...


[*********************100%***********************]  1 of 1 completed


data/samsung_2023_2024.csv 저장 완료!
images/04_stl_decomposition.png 저장 성공!
모든 그래프 생성 및 images/ 폴더 저장 완료!
